# Notebook 02 — Softmax and temperature

Paired with `theory/02_softmax.md`. **Mode:** theory-first — read the chapter end-to-end and write the §2.7 predictions before running anything.

## Question, Hypothesis, Method, Prediction, Confidence

**Question.** Verify the core structural properties of softmax and probe its temperature / scaling behavior empirically:
1. Shift invariance and numerical stability (Prop 2.2).
2. Entropy of $\text{softmax}_T(s)$ across $T \in [10^{-2}, 10^2]$ — asymptotes and transition.
3. Saturation margin: for fixed $n$, what $\Delta$ reaches $p_{\text{win}} = 0.99$? Does it scale as $\log n$ (Corollary 2.5.1)?
4. Entropy heatmap over $(\sigma, n)$ for $s \sim \mathcal{N}(0, \sigma^2 I_n)$ — is there a combined quantity that controls entropy (Exercise 2.5)?
5. Softmax Jacobian spectrum — is it PSD (Exercise 2.1)? What is its rank?
6. LSE-minus-max gap as a function of $n$ for random scores — does it saturate the $\log n$ upper bound?

**Hypothesis.** Propositions 2.2, 2.3, 2.5, 2.6 all hold exactly; the empirical work is about *shape* and *scaling constants*, not about whether the props are right. The interesting updates come from (2), (4), and (6) — where the answer has free parameters not pinned by the propositions.

**Method.** Six small experiments in numpy + torch, all computation on CPU in well under a minute.

**Prediction.** *Paste your §2.7 predictions here.*

1. Shift-invariance error (safe vs. naive) — 
2. Entropy vs. temperature shape — 
3. Saturation margin $\Delta^\star(n)$ for $p = 0.99$ — 
4. Entropy heatmap — level-set geometry — 
5. Jacobian rank and PSD — 
6. LSE-minus-max vs. $n$ scaling — 

**Confidence.** Overall: *[low / medium / high, reason]*.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)
np.set_printoptions(precision=4, suppress=True)

---
## Experiment 1 — shift invariance and numerical stability

`softmax_safe` subtracts the max before exponentiating (Prop 2.2). `softmax_naive` does not. Run both on $s = (1, 2, 3)$ and on $s + 1000 \cdot \mathbf{1}$; compare.

In [ ]:
def softmax_naive(s):
    e = np.exp(s)
    return e / e.sum()

def softmax_safe(s):
    s = s - s.max()
    e = np.exp(s)
    return e / e.sum()

s = np.array([1.0, 2.0, 3.0])
shifted = s + 1000.0

print('s = (1, 2, 3):')
print(f'  naive:  {softmax_naive(s)}')
print(f'  safe:   {softmax_safe(s)}')
print('\ns + 1000:')
with np.errstate(over='ignore', invalid='ignore'):
    n_result = softmax_naive(shifted)
print(f'  naive:  {n_result}   # overflows')
print(f'  safe:   {softmax_safe(shifted)}')
print(f'\nmax-abs diff between safe(s) and safe(s+1000): {np.max(np.abs(softmax_safe(s) - softmax_safe(shifted))):.2e}')

**Sub-finding 1.** *Did the naive form overflow as predicted? Was `safe(s)` exactly equal to `safe(s+1000)`, or did you see floating-point drift?*

---
## Experiment 2 — entropy vs. temperature

Fix $s \sim \mathcal{N}(0, I_{50})$. Compute $H(\text{softmax}_T(s))$ for $T \in [10^{-2}, 10^2]$.

In [ ]:
n = 50
s = np.random.randn(n)
Ts = np.logspace(-2, 2, 200)

def entropy(p):
    return -np.sum(p * np.log(p + 1e-30))

Hs = [entropy(softmax_safe(s / T)) for T in Ts]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(Ts, Hs)
ax.axhline(np.log(n), color='k', linestyle=':', alpha=0.5, label=f'log(n) = {np.log(n):.2f}  (uniform)')
ax.axhline(0, color='r', linestyle=':', alpha=0.5, label='0  (one-hot)')
ax.set_xscale('log')
ax.set_xlabel('temperature T')
ax.set_ylabel('H(softmax_T(s))  (nats)')
ax.set_title(f'Entropy vs temperature  (n = {n}, fixed s ~ N(0, I))')
ax.legend()
plt.tight_layout()
plt.show()

**Sub-finding 2.** *Did entropy start near 0 (low $T$, one-hot) and saturate at $\log n$ (high $T$, uniform)? Where is the inflection (the "effective" temperature for this $s$)? Does that inflection location correlate with $\|s\|_\infty$ or $\|s\|_2$?*

---
## Experiment 3 — saturation margin (Corollary 2.5.1)

For each $n \in \{4, 16, 64, 256, 1024\}$, find the minimum margin $\Delta$ for which $p_{\text{win}} = e^\Delta / (e^\Delta + n - 1) \ge 0.99$. Compare the numerical answer against the closed-form $\Delta^\star = \log((n-1) \cdot 99)$.

In [ ]:
def p_win(delta, n):
    return np.exp(delta) / (np.exp(delta) + n - 1)

ns = [4, 16, 64, 256, 1024]
target = 0.99
print(f'{"n":>6}  {"Δ_numerical":>14}  {"Δ_closed-form":>16}  {"log(n-1)":>10}')
print('-' * 52)
for n in ns:
    # closed form from p = 1/(1 + (n-1) e^-Δ) => Δ = log((n-1) * p / (1-p))
    delta_closed = np.log((n - 1) * target / (1 - target))
    # numerical: bisection
    lo, hi = 0.0, 100.0
    while hi - lo > 1e-6:
        mid = (lo + hi) / 2
        if p_win(mid, n) < target:
            lo = mid
        else:
            hi = mid
    print(f'{n:>6}  {lo:>14.4f}  {delta_closed:>16.4f}  {np.log(n - 1):>10.4f}')

# Plot: Δ^star as a function of n
ns_plot = np.logspace(0.5, 4, 50).astype(int)
deltas = [np.log((n - 1) * target / (1 - target)) for n in ns_plot]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ns_plot, deltas, 'o-')
ax.plot(ns_plot, np.log(ns_plot), ':', alpha=0.5, label='log(n)')
ax.set_xscale('log')
ax.set_xlabel('n (number of keys)')
ax.set_ylabel(f'Δ* for p_win = {target}')
ax.set_title('Saturation margin scales as log(n)')
ax.legend()
plt.tight_layout()
plt.show()

**Sub-finding 3.** *Did the margin grow as $\log n$ as Corollary 2.5.1 predicted? The plot should be near-linear on log-x — is it?*

---
## Experiment 4 — entropy heatmap over $(\sigma, n)$

For $s \sim \mathcal{N}(0, \sigma^2 I_n)$, compute mean entropy of $\text{softmax}(s)$ over many samples, for a grid of $(\sigma, n)$. Exercise 2.5 asks: what combined quantity controls entropy?

In [ ]:
sigmas = np.logspace(-1, 1.5, 20)     # 0.1 to ~30
ns = np.unique(np.logspace(0.7, 3, 20).astype(int))  # ~5 to 1000
n_trials = 200

H_grid = np.zeros((len(sigmas), len(ns)))
for i, sigma in enumerate(sigmas):
    for j, n in enumerate(ns):
        S = sigma * np.random.randn(n_trials, n)
        # row-wise softmax, row-wise entropy
        S = S - S.max(axis=-1, keepdims=True)
        P = np.exp(S); P /= P.sum(axis=-1, keepdims=True)
        H = -np.sum(P * np.log(P + 1e-30), axis=-1)
        H_grid[i, j] = H.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Raw entropy heatmap
im = axes[0].imshow(H_grid, aspect='auto', origin='lower',
                    extent=[np.log10(ns[0]), np.log10(ns[-1]), np.log10(sigmas[0]), np.log10(sigmas[-1])],
                    cmap='viridis')
axes[0].set_xlabel('log10(n)')
axes[0].set_ylabel('log10(σ)')
axes[0].set_title('Mean entropy H(softmax(s))')
plt.colorbar(im, ax=axes[0])

# Entropy normalized by log(n) — saturation fraction
H_frac = H_grid / np.log(ns)[None, :]
im2 = axes[1].imshow(H_frac, aspect='auto', origin='lower',
                     extent=[np.log10(ns[0]), np.log10(ns[-1]), np.log10(sigmas[0]), np.log10(sigmas[-1])],
                     cmap='plasma', vmin=0, vmax=1)
axes[1].set_xlabel('log10(n)')
axes[1].set_ylabel('log10(σ)')
axes[1].set_title('H / log(n)  — fraction of maximum entropy')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

**Sub-finding 4.** *Look at the level sets in the left heatmap — are they vertical, horizontal, or diagonal? Which single combined quantity of $(\sigma, n)$ controls entropy? (Hint: try $\sigma \sqrt{\log n}$ — the maximum of $n$ i.i.d. $\mathcal{N}(0, \sigma^2)$ scales this way.) What does the right plot (normalized entropy) show?*

---
## Experiment 5 — softmax Jacobian spectrum

Exercise 2.1: the softmax Jacobian is the covariance matrix of the categorical distribution $p$. Covariance matrices are PSD. Verify the Jacobian is PSD and determine its rank.

In [ ]:
n = 8
s = torch.randn(n)
p = F.softmax(s, dim=-1).numpy()

J = np.diag(p) - np.outer(p, p)
eigs = np.linalg.eigvalsh(J)

print(f'Softmax Jacobian eigenvalues (sorted):\n{eigs}')
print(f'\nMin eigenvalue: {eigs.min():.4e}   (should be ~0, confirming PSD not PD)')
print(f'Rank (eigenvalues > 1e-10): {np.sum(eigs > 1e-10)}   (expected: n - 1 = {n - 1})')
print(f'\nNull vector: the all-ones vector.')
print(f'J @ ones: {J @ np.ones(n)}   (should be zero — Corollary 2.3.1)')

**Sub-finding 5.** *Jacobian confirmed PSD? Rank is exactly $n - 1$ and the null space is the all-ones direction — why? (Hint: mass-conservation. Perturbing all $s_i$ by the same constant doesn't change $p$ — Prop 2.2.)*

---
## Experiment 6 — LSE minus max, scaling with $n$

Prop 2.6 gives $\max_i s_i \le \text{LSE}(s) \le \max_i s_i + \log n$. The worst case is when all $s_i$ are equal. For random $s \sim \mathcal{N}(0, I_n)$, how close does the gap get to $\log n$?

In [ ]:
ns = np.unique(np.logspace(0.5, 4, 30).astype(int))
n_trials = 1000
gaps = []
for n in ns:
    S = np.random.randn(n_trials, n)
    lse = np.log(np.exp(S - S.max(axis=-1, keepdims=True)).sum(axis=-1)) + S.max(axis=-1)
    gap = lse - S.max(axis=-1)
    gaps.append(gap.mean())

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ns, gaps, 'o-', label='mean(LSE − max)')
ax.plot(ns, np.log(ns), ':', alpha=0.5, label='log(n)  (worst-case bound)')
ax.set_xscale('log')
ax.set_xlabel('n')
ax.set_ylabel('gap (nats)')
ax.set_title('LSE − max for random Gaussian scores')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Gap at n=10:   {gaps[np.searchsorted(ns, 10)]:.3f}   (log 10 = {np.log(10):.3f})')
print(f'Gap at n=1000: {gaps[np.searchsorted(ns, 1000)]:.3f}   (log 1000 = {np.log(1000):.3f})')

**Sub-finding 6.** *Did the gap grow like $\log n$ (saturating the upper bound) or much slower? For Gaussian $s$, the max is concentrated around $\sqrt{2 \log n}$ — does that explain the gap behavior? What is the asymptotic rate?*

---
## Finding

*Write 3–6 sentences:*

- Which predictions held?
- Which missed? Mental-model update in one sentence.
- Did Experiment 4 reveal the conjectured combined quantity for Exercise 2.5(c)? What does the level-set geometry say?
- What's the next question? Candidate: in attention, pre-softmax scores have $\sigma \approx \sqrt{d_k}$ at init (Ch. 3 preview) — where does that put us on the heatmap, and does it explain why unscaled attention saturates?

*Append 2–3 sentences to `learnings.md`.*